# Tier 21: aerodynamics build-up

User direction (2026-10-02): lean heavily on AeroSandbox's built-in aerodynamics. `asb.AeroBuildup` on the
`Aircraft.to_asb()` geometry is the primary model; Scholz's level-0 method (HAW Hamburg, *Aircraft Design*
ch. 13 "Drag Prediction") fills only what AeroBuildup does not provide and is an independent hand check.
`SimpleAerodynamics` stays as the simplest model and remains the Halo default (plan 025).

| Model | Lift | Parasite drag | Induced drag | Extras |
|---|---|---|---|---|
| `SimpleAerodynamics` (Tier 5) | linear, wing only | Cf x FF x S_wet + 0.8 m2 guessed misc. | Nita-Scholz e | constant 7 % download |
| `BuildupAerodynamics` (this tier) | AeroBuildup (NeuralFoil sections, tails, post-stall) | AeroBuildup components incl. nacelle bodies, x Scholz Q; 3.00 ft2 fittings (XV-15, NDARC) | AeroBuildup span efficiency x k_e,F x k_e,M | blown wing, geometric download, transition at 10 % chord |
| `ScholzAerodynamics` (hand check) | linear | Scholz 13.15-13.24, Torenbeek S_wet, Table 13.4 Q | Nita-Scholz e (turboprop k_e,D0) | Korn-Lock wave drag |

Sources: Scholz ch. 13 (fetched 2026-10-03); Nita and Scholz, DLRK 2012; Korn-Lock as in AeroSandbox
`transonic.Cd_wave_Korn` (Mason); W. Johnson, "NDARC Validation and Demonstration", AHS 2010, Table 1 (XV-15
D/q and component breakdown); NASA TM X-62407 (XV-15 7 % download with flaps).

```powershell
uv run jupyter lab notebooks/tier21_aerodynamics
```

In [ ]:
import sys
import time
from dataclasses import replace
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
for path in (repo_root / "src", repo_root):     # this checkout's package even if another one is installed
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import aerosandbox as asb
import aerosandbox.numpy as np
import aerosandbox.tools.units as u
import matplotlib.pyplot as plt

from aircraft_closure.aerodynamics.buildup import BuildupAerodynamics
from aircraft_closure.aerodynamics.download import HoverDownload, download_fraction
from aircraft_closure.aerodynamics.scholz import (InterferenceFactors, ScholzAerodynamics, form_factor_nacelle,
                                                  oswald_nita_scholz, skin_friction_turbulent)
from aircraft_closure.aerodynamics.slipstream import BlownWing, RotorState
from examples.halo_sizing import (HaloAssumptions, HaloRequirements, build_halo_aerodynamics, build_halo_aircraft,
                                  solve_halo_sizing)
from examples.trajectory_optimization import halo_trajectory_case, solve_min_energy_transition, solve_min_time_climb
from examples.xv15_reference import Xv15Reference, build_xv15_aircraft

check_log = []


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


def scalar(x):
    return float(np.asarray(x).ravel()[0])


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua, yellow = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8,
                     "axes.labelcolor": ink, "xtick.color": ink_muted, "ytick.color": ink_muted})
colors = {"simple": ink_muted, "buildup": blue, "scholz": orange}
requirements = HaloRequirements()
models = {m: build_halo_aerodynamics(requirements, replace(HaloAssumptions(), aerodynamics_model=m))
          for m in ("simple", "buildup", "scholz")}

## 1. The reference aircraft

The plan 022 reference (780 kg payload, 14,436 lb, equivalent-circuit battery, `SimpleAerodynamics`) is sized
once and its numeric design is used for the polars below.

In [ ]:
t0 = time.time()
reference = solve_halo_sizing()
time_simple_s = time.time() - t0
aircraft = build_halo_aircraft(reference.design)
print(f"reference: {reference.mass_takeoff_kg / u.lbm:,.0f} lb, wing {reference.design.area_wing_m2:.2f} m2, "
      f"span {reference.span_m:.2f} m, rotor R {reference.radius_rotor_m:.2f} m, cruise "
      f"{reference.velocity_cruise_m_s / u.knot:.0f} kt at CL {reference.cl_cruise:.2f}, L/D {reference.lift_to_drag_cruise:.2f}"
      f" ({time_simple_s:.0f} s)")
check("reference unchanged: 14,436 lb with SimpleAerodynamics", abs(reference.mass_takeoff_kg / u.lbm - 14436) < 5)
check("nacelles exported as two AeroSandbox bodies", len(aircraft.to_asb().fuselages) == 3)

## 2. Closed-form pieces (Scholz, Korn, Nita-Scholz)

- Turbulent flat plate with Mach (Scholz 13.17): `0.455 / (log10 Re)^2.58 / (1 + 0.144 M^2)^0.65`.
- Raymer nacelle form factor (13.24): `1 + 0.35 / (l/d)`; the Halo nacelles are 9 ft x 3.3 ft.
- Korn with Lock's criterion: `M_DD = 0.87 - t/c - CL/10`, `M_crit = M_DD - (0.1/80)^(1/3)`. The 23 % thick
  NACA 2423 wing at CL 0.6 has `M_crit` about 0.43, above the 210 kt / 10,000 ft Mach of 0.33: no drag rise.
- Nita-Scholz Oswald factor: `e = e_theo k_e,F k_e,D0 k_e,M`.

In [ ]:
check("Scholz Cf at M = 0 is 0.455 / (log Re)^2.58",
      abs(scalar(skin_friction_turbulent(1e7, 0.0)) - 0.455 / 7**2.58) < 1e-12)
fineness = HaloAssumptions().length_nacelle_m / HaloAssumptions().diameter_nacelle_m
check(f"Raymer nacelle FF = 1 + 0.35 / {fineness:.2f} = {scalar(form_factor_nacelle(fineness)):.3f}",
      abs(scalar(form_factor_nacelle(fineness)) - (1 + 0.35 / fineness)) < 1e-12)
thickness = aircraft.wing.airfoil.max_thickness()
mach_cruise = scalar(210 * u.knot / asb.Atmosphere(10000 * u.foot).speed_of_sound())
mach_crit = 0.87 - thickness - 0.6 / 10 - (0.1 / 80)**(1 / 3)
print(f"wing t/c {thickness:.3f}; M_crit {mach_crit:.3f} at CL 0.6; M at 210 kt, 10,000 ft {mach_cruise:.3f}")
check("Korn: no wave drag at 210 kt on the 23 % section",
      scalar(models["scholz"].wave_drag_coefficient(aircraft, 0.6, mach_cruise)) == 0.0)
check("Korn: drag rise 20 (M - M_crit)^4 above M_crit",
      abs(scalar(models["scholz"].wave_drag_coefficient(aircraft, 0.6, mach_crit + 0.1)) - 20 * 0.1**4) < 1e-6)
e_halo = scalar(oswald_nita_scholz(1.0, aircraft.wing.aspect_ratio,
                                   aircraft.fuselage.diameter_m / scalar(aircraft.wing.span_m()), 0.25))
print(f"Nita-Scholz e (AR {aircraft.wing.aspect_ratio}, d_F/b {aircraft.fuselage.diameter_m / scalar(aircraft.wing.span_m()):.3f}): {e_halo:.3f}")
check("Nita-Scholz e in the usual 0.7-0.8 band for an unswept AR 6 wing", 0.7 < e_halo < 0.8)

## 3. Drag polars at the cruise condition

Polars at 10,000 ft and 150 kt for the three models on the reference geometry. AeroBuildup's lift includes the
tails at zero incidence (no downwash), so it has a higher slope than the wing-only linear models; at a given
CL the drag is the comparable quantity.

In [ ]:
altitude_m, velocity_m_s = 10000 * u.foot, 150 * u.knot
alphas = np.linspace(-2, 14, 17)
polars = {}
for name, aero in models.items():
    t0 = time.time()
    results = [aero.evaluate(aircraft, velocity_m_s, altitude_m, a) for a in alphas]
    polars[name] = dict(cl=np.array([scalar(r.cl) for r in results]), cd=np.array([scalar(r.cd) for r in results]),
                        cd0=np.array([scalar(r.cd0) for r in results]),
                        e=np.array([scalar(r.oswald_efficiency) for r in results]))
    print(f"{name:8s} {1e3 * (time.time() - t0) / len(alphas):6.1f} ms per point")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, p in polars.items():
    axes[0].plot(p["cd"], p["cl"], color=colors[name], marker="o", ms=3, label=name)
    axes[1].plot(p["cl"], p["cl"] / p["cd"], color=colors[name], marker="o", ms=3, label=name)
axes[0].set(xlabel="CD", ylabel="CL", title="Drag polar, 150 kt, 10,000 ft")
axes[1].set(xlabel="CL", ylabel="L/D", title="Lift-to-drag ratio")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

def at_cl(name, cl_target, key):
    p = polars[name]
    return float(np.interp(cl_target, p["cl"], p[key]))

cl_cruise = 0.6
for name in models:
    cd = at_cl(name, cl_cruise, "cd")
    print(f"{name:8s} at CL {cl_cruise}: CD {cd:.4f}, CD0 {at_cl(name, cl_cruise, 'cd0'):.4f}, "
          f"e {at_cl(name, cl_cruise, 'e'):.3f}, L/D {cl_cruise / cd:.2f}")
check("AeroBuildup and Scholz CD within 10 % at CL 0.6",
      abs(at_cl("buildup", cl_cruise, "cd") / at_cl("scholz", cl_cruise, "cd") - 1) < 0.10)
check("both build-ups below SimpleAerodynamics with the guessed 0.8 m2",
      at_cl("buildup", cl_cruise, "cd") < at_cl("simple", cl_cruise, "cd")
      and at_cl("scholz", cl_cruise, "cd") < at_cl("simple", cl_cruise, "cd"))
check("polars monotonic: CD rises with CL above CL 0.2",
      all(np.all(np.diff(p["cd"][p["cl"] > 0.2]) > 0) for p in polars.values()))

## 4. Drag breakdown at cruise

Component parasite drag at CL about 0.6 (AeroBuildup at the matching angle of attack). The 3.00 ft2
"fittings and fixtures" area is the XV-15 value from NDARC; it replaces the guessed 0.8 m2 (8.6 ft2), which was
almost the XV-15's whole drag area.

In [ ]:
buildup = models["buildup"]
alpha_cruise = float(np.interp(cl_cruise, polars["buildup"]["cl"], alphas))
b_items = {i.label: scalar(i.cd0) for i in buildup.evaluate_buildup(aircraft, velocity_m_s, altitude_m,
                                                                   alpha_cruise).breakdown}
s_items = {i.label: scalar(i.cd0) for i in models["scholz"].parasite_drag_breakdown(aircraft, velocity_m_s, altitude_m)}
m_items = {i.label: scalar(i.cd0) for i in models["simple"].parasite_drag_breakdown(aircraft, velocity_m_s, altitude_m)}
labels = ["wing", "horizontal_tail", "vertical_tail", "fuselage", "nacelles", "miscellaneous"]
area_ft2 = aircraft.wing.area_m2 / u.foot**2
print(f"{'item':16s} {'simple':>8s} {'buildup':>8s} {'scholz':>8s}   (D/q ft2: buildup / scholz)")
for label in labels:
    print(f"{label:16s} {m_items.get(label, 0):8.4f} {b_items.get(label, 0):8.4f} {s_items.get(label, 0):8.4f}"
          f"   {b_items.get(label, 0) * area_ft2:5.2f} / {s_items.get(label, 0) * area_ft2:5.2f}")
totals = [sum(d.get(l, 0) for l in labels) for d in (m_items, b_items, s_items)]
print(f"{'total':16s} {totals[0]:8.4f} {totals[1]:8.4f} {totals[2]:8.4f}   {totals[1] * area_ft2:5.2f} / {totals[2] * area_ft2:5.2f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
width = 0.27
x = np.arange(len(labels))
for k, (name, items) in enumerate((("simple", m_items), ("buildup", b_items), ("scholz", s_items))):
    ax.bar(x + (k - 1) * width, [items.get(l, 0) for l in labels], width, color=colors[name], label=name)
ax.set_xticks(x, [l.replace("_", " ") for l in labels])
ax.set(ylabel="CD0 contribution (on S_wing)", title="Parasite drag breakdown, 150 kt, 10,000 ft")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
check("nacelle drag: AeroBuildup body x Q 1.5 within 35 % of Scholz (Raymer FF)",
      abs(b_items["nacelles"] / s_items["nacelles"] - 1) < 0.35)
check("wing profile drag within 25 % (NeuralFoil, transition 10 % vs fully turbulent Cf x FF)",
      abs(b_items["wing"] / s_items["wing"] - 1) < 0.25)

## 5. XV-15 check against NDARC

Johnson (2010) Table 1 gives the XV-15 cruise drag area D/q = 9.25 ft2: fuselage 1.56, fuselage fittings and
fixtures 3.00, pylons 2 x 0.76, horizontal tail 0.63, vertical tail 0.36, wing 2.18. The components modelled
here (wing, tails, fuselage, nacelles) sum to 6.25 ft2. The XV-15 aircraft of Tier 10a with 9 ft x 3.3 ft tip
nacelles and an H-tail (Q = 1.08) is evaluated at its 200 kt / 20,000 ft cruise near zero lift.

In [ ]:
xv = Xv15Reference()
xv15 = build_xv15_aircraft(xv, mass_engine_kg=300.0)
xv15 = replace(xv15, nacelles=replace(xv15.nacelles, length_m=9 * u.foot, diameter_m=3.3 * u.foot,
                                      y_m=xv15.wing.span_m() / 2))
h_tail = InterferenceFactors(horizontal_tail=1.08, vertical_tail=1.08)
ndarc = dict(wing=2.18, tails=0.99, fuselage=1.56, nacelles=1.52)
for name, aero in (("buildup", BuildupAerodynamics(interference=h_tail)),
                   ("scholz", ScholzAerodynamics(interference=h_tail))):
    if name == "buildup":
        items = aero.evaluate_buildup(xv15, xv.velocity_cruise_m_s, xv.altitude_cruise_m, 0.0).breakdown
    else:
        items = aero.parasite_drag_breakdown(xv15, xv.velocity_cruise_m_s, xv.altitude_cruise_m)
    d = {i.label: scalar(i.cd0) * xv15.wing.area_m2 / u.foot**2 for i in items}
    grouped = dict(wing=d["wing"], tails=d["horizontal_tail"] + d["vertical_tail"], fuselage=d["fuselage"],
                   nacelles=d["nacelles"])
    total = sum(grouped.values())
    print(f"{name:8s} " + ", ".join(f"{k} {v:.2f}" for k, v in grouped.items()) + f"; sum {total:.2f} ft2 "
          f"(NDARC 6.25), with 3.00 fittings {total + 3.0:.2f} (NDARC 9.25)")
    check(f"XV-15 component D/q ({name}) within 25 % of NDARC", abs(total / 6.25 - 1) < 0.25)

## 6. Blown wing

The rotor slipstream in airplane mode: momentum-theory induced velocity, its development to the wing
(`v_i (1 + x / sqrt(x^2 + R^2))`, x = 0.4 R), a contracted slipstream radius, and the ideal swirl
`T / (rho A Omega 2R/3)`. On the immersed span (half of each tip rotor's slipstream): the dynamic-pressure ratio
raises lift and profile drag; the swirl angle raises the local angle of attack (inboard-up rotation) and the wing
recovers half of the swirl kinetic-energy flux as thrust. All increments vanish at zero thrust.

In [ ]:
alpha_blown = 4.0
speed_rotor_rad_s = 0.55 * reference.design.speed_tip_m_s / reference.radius_rotor_m
thrusts_N = np.linspace(0, 15000, 16)
blown_aero = models["buildup"]
rows = [blown_aero.evaluate_buildup(aircraft, velocity_m_s, altitude_m, alpha_blown,
                                    rotor_state=RotorState(t, speed_rotor_rad_s)) for t in thrusts_N]
delta_cl = np.array([scalar(r.blown.delta_cl) for r in rows])
delta_cd = np.array([scalar(r.blown.delta_cd_profile + r.blown.delta_cd_swirl) for r in rows])
ratio_q = np.array([scalar(r.blown.ratio_dynamic_pressure) for r in rows])
swirl_deg = np.array([scalar(r.blown.angle_swirl_deg) for r in rows])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(thrusts_N / 1e3, delta_cl, color=blue, label="delta CL")
axes[0].plot(thrusts_N / 1e3, 10 * delta_cd, color=orange, label="10 x delta CD (profile + swirl)")
axes[0].set(xlabel="thrust per rotor (kN)", ylabel="increment on S_wing", title=f"Blown-wing increments, alpha {alpha_blown} deg")
axes[0].legend(frameon=False)
axes[1].plot(thrusts_N / 1e3, ratio_q, color=aqua, label="q_slipstream / q")
ax2 = axes[1].twinx()
ax2.plot(thrusts_N / 1e3, swirl_deg, color=yellow, label="swirl angle (deg)")
axes[1].set(xlabel="thrust per rotor (kN)", ylabel="dynamic-pressure ratio", title="Slipstream at the wing")
ax2.set_ylabel("swirl angle (deg)")
axes[1].legend(loc="upper left", frameon=False); ax2.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()
cruise_thrust_N = 0.5 * scalar(blown_aero.evaluate(aircraft, velocity_m_s, altitude_m, alpha_blown).drag_N)
cruise_row = blown_aero.evaluate_buildup(aircraft, velocity_m_s, altitude_m, alpha_blown,
                                         rotor_state=RotorState(cruise_thrust_N, speed_rotor_rad_s))
print(f"at the cruise thrust {cruise_thrust_N:.0f} N per rotor: q ratio {scalar(cruise_row.blown.ratio_dynamic_pressure):.4f}, "
      f"swirl {scalar(cruise_row.blown.angle_swirl_deg):.2f} deg, immersed {scalar(cruise_row.blown.area_immersed_m2):.1f} m2 "
      f"of {aircraft.wing.area_m2:.1f}, delta CL {scalar(cruise_row.blown.delta_cl):+.4f}, delta CD "
      f"{scalar(cruise_row.blown.delta_cd_profile + cruise_row.blown.delta_cd_swirl):+.5f}")
check("zero thrust: no increment", abs(delta_cl[0]) < 1e-12 and abs(delta_cd[0]) < 1e-12)
check("lift increment grows with thrust", np.all(np.diff(delta_cl) > 0))
check("swirl recovery: net drag increment negative and growing with thrust", np.all(np.diff(delta_cd) < 0))

## 7. Hover download from geometry

`DL/T = C_D,v S_immersed / A_disk` with the fully developed wake (NDARC), the immersed strip under the inboard
half of each contracted wake (radius R/sqrt 2), and the flap-projected chord. C_D,v = 0.846 reproduces the
XV-15's 7 % with flaps; the Halo reference geometry gives about the same.

In [ ]:
xv15_chord_m = 168.88 / 32.17 * u.foot
dl_xv15 = scalar(download_fraction(xv15_chord_m, 12.5 * u.foot, 2))
dl_halo = scalar(models["buildup"].hover_download_fraction(aircraft))
print(f"download: XV-15 {dl_xv15:.4f}, Halo reference {dl_halo:.4f} (constant 0.07 in SimpleAerodynamics)")
check("XV-15 calibration reproduces 0.07", abs(dl_xv15 - 0.07) < 2e-4)
flaps = np.linspace(0, 90, 10)
chord_m = aircraft.wing.area_m2 / scalar(aircraft.wing.span_m())
dl_flap = [scalar(download_fraction(chord_m, reference.radius_rotor_m, 2, HoverDownload(deflection_flap_hover_deg=f)))
           for f in flaps]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(flaps, dl_flap, color=blue)
ax.set(xlabel="hover flap deflection (deg)", ylabel="download DL/T", title="Halo download vs flap setting")
plt.tight_layout()
plt.show()
check("flap deflection lowers download", np.all(np.diff(dl_flap) < 0))

## 8. Halo sized with each model

The same requirements (780 kg, 445 nm, 210 kt at 10,000 ft) and assumptions; only `aerodynamics_model`
changes. The build-up runs AeroBuildup (NeuralFoil) at every airplane-mode point inside the coupled solve.

In [ ]:
sized, times = {"simple": reference}, {"simple": time_simple_s}
for name, extra in (("buildup", {}), ("buildup, no blowing", dict(blown_wing=False)), ("scholz", {})):
    t0 = time.time()
    a = replace(HaloAssumptions(), aerodynamics_model=name.split(",")[0], **extra)
    sized[name] = solve_halo_sizing(assumptions=a)
    times[name] = time.time() - t0
print(f"{'model':22s} {'MTOM lb':>8s} {'L/D':>6s} {'CL':>6s} {'V kt':>6s} {'wing m2':>8s} {'fuel kg':>8s} {'kWh':>6s} {'solve s':>8s}")
for name, r in sized.items():
    print(f"{name:22s} {r.mass_takeoff_kg / u.lbm:8.0f} {r.lift_to_drag_cruise:6.2f} {r.cl_cruise:6.3f} "
          f"{r.velocity_cruise_m_s / u.knot:6.0f} {r.design.area_wing_m2:8.2f} {r.mass_fuel_kg:8.0f} "
          f"{r.energy_battery_kWh:6.1f} {times[name]:8.0f}")
check("buildup sizing closes with all margins", sized["buildup"].min_margin > -1e-6)
check("buildup lighter than the SimpleAerodynamics reference",
      sized["buildup"].mass_takeoff_kg < reference.mass_takeoff_kg)
check("Scholz hand check within 1 % of the buildup take-off mass",
      abs(sized["scholz"].mass_takeoff_kg / sized["buildup"].mass_takeoff_kg - 1) < 0.01)
check("blown wing lowers the take-off mass", sized["buildup"].mass_takeoff_kg < sized["buildup, no blowing"].mass_takeoff_kg)

## 9. Trajectories with the build-up

The Tier 14 problems on the build-up-sized aircraft with `BuildupAerodynamics` (unblown polar; the blown wing in
conversion is deferred). AeroBuildup is evaluated vectorized over the collocation nodes.

In [ ]:
case = halo_trajectory_case(sized["buildup"], HaloRequirements(), replace(HaloAssumptions(), aerodynamics_model="buildup"))
t0 = time.time()
transition = solve_min_energy_transition(case)
time_transition_s = time.time() - t0
t0 = time.time()
climb = solve_min_time_climb(case)
time_climb_s = time.time() - t0
print(f"minimum-energy transition: {transition.duration_s:.1f} s, {transition.energy_bus_J[-1] / 3.6e6:.2f} kWh "
      f"({time_transition_s:.0f} s solve); time to climb to 10,000 ft: {climb.duration_s:.0f} s ({time_climb_s:.0f} s solve)")
check("transition solves with the build-up", 10 < transition.duration_s < 40)
check("climb solves with the build-up", 100 < climb.duration_s < 400)

## 10. Summary

In [ ]:
failed = [name for name, passed in check_log if not passed]
print(f"{len(check_log) - len(failed)} / {len(check_log)} notebook checks passed")
for name in failed:
    print("  FAIL:", name)

In [ ]:
import os, unittest

os.chdir(repo_root)
suite = unittest.defaultTestLoader.discover(str(repo_root / "tests"), top_level_dir=str(repo_root / "tests"))
test_result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
assert test_result.wasSuccessful() and not failed, "Verification failed"